In [2]:
import csv
from rdkit import Chem
from rdkit.Chem import AllChem

# Function to attach substituents to the cubane core
def attach_substituents(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            raise ValueError(f"Invalid SMILES: {smiles}")

        # Split the molecule into fragments
        fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        cubane_core = max(fragments, key=lambda x: x.GetNumAtoms())  # Largest fragment as core

        # Identify substituents (smaller fragments)
        substituents = [frag for frag in fragments if frag != cubane_core]

        # Editable version of the cubane core
        rw_cubane = Chem.RWMol(cubane_core)

        # Find available positions (carbons with degree < 4)
        attachment_points = [
            atom.GetIdx() for atom in rw_cubane.GetAtoms()
            if atom.GetSymbol() == "C" and atom.GetDegree() < 4
        ]

        # Attach each substituent to an available position
        for sub_mol in substituents:
            if not attachment_points:
                break  # Stop if no more positions are available

            # Combine the substituent with the cubane core
            rw_combined = Chem.CombineMols(rw_cubane, sub_mol)
            rw_combined = Chem.RWMol(rw_combined)

            # Attach the substituent to the first available position
            attachment_idx = attachment_points.pop(0)  # Take the first available position
            sub_attachment_idx = rw_combined.GetNumAtoms() - sub_mol.GetNumAtoms()  # First atom of the substituent
            rw_combined.AddBond(attachment_idx, sub_attachment_idx, Chem.BondType.SINGLE)

            # Update the molecule
            rw_cubane = Chem.RWMol(rw_combined)

        # Return the fixed SMILES
        return Chem.MolToSmiles(rw_cubane)
    except Exception as e:
        print(f"Error processing SMILES {smiles}: {e}")
        return None

# Function to process a CSV file with SMILES and save corrected SMILES
def process_smiles_csv(input_file, output_file):
    corrected_data = []

    # Read the input CSV file
    with open(input_file, "r") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            original_smiles = row["SMILES"]
            corrected_smiles = attach_substituents(original_smiles)
            corrected_data.append({"Original_SMILES": original_smiles, "Corrected_SMILES": corrected_smiles})

    # Write the corrected SMILES to a new CSV file
    with open(output_file, "w", newline="") as csvfile:
        fieldnames = ["Original_SMILES", "Corrected_SMILES"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(corrected_data)

    print(f"Corrected SMILES saved to {output_file}")

# File paths
input_csv = "filtered1700molecules.csv"  # Replace with your input CSV file containing SMILES
output_csv = "corrected_smiles.csv"  # Output file to save corrected SMILES

# Process the SMILES
process_smiles_csv(input_csv, output_csv)


Corrected SMILES saved to corrected_smiles.csv


In [4]:
import pandas as pd
from rdkit import Chem

# Load Excel file
df = pd.read_excel("firstcorrected1700molecules.xlsx")

# Check if the column name is correct
column_name = "SMILES"  # Change this if your column name is different

if column_name not in df.columns:
    print(f"❌ Column '{column_name}' not found in Excel! Check column names:", df.columns)
else:
    # Validate SMILES
    def is_valid_smiles(smiles):
        return Chem.MolFromSmiles(smiles) is not None

    df_valid = df[df[column_name].apply(is_valid_smiles)].copy()

    # Save only valid SMILES to a new Excel file
    df_valid.to_excel("valid_smiles.xlsx", index=False)
    print(f"✅ Valid SMILES saved to 'valid_smiles.xlsx'. Total valid entries: {len(df_valid)}")


[11:26:12] Explicit valence for atom # 21 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 18 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 9 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 9 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 8 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 18 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 18 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 21 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 18 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 9 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 23 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 14 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 14 O, 3, is greater than permitted
[11:26:12] Explicit valence for atom # 12 

✅ Valid SMILES saved to 'valid_smiles.xlsx'. Total valid entries: 988


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED, GraphDescriptors, AllChem

def compute_descriptors(mol):
    desc = {}
    
    # Compute Gasteiger charges
    try:
        AllChem.ComputeGasteigerCharges(mol)
        charges = [atom.GetDoubleProp('_GasteigerCharge') for atom in mol.GetAtoms() if atom.HasProp('_GasteigerCharge')]
        desc['MaxPartialCharge'] = max(charges) if charges else None
        desc['MinPartialCharge'] = min(charges) if charges else None
    except:
        desc['MaxPartialCharge'] = None
        desc['MinPartialCharge'] = None

    # BCUT2D descriptors
    try:
        bcut = rdMolDescriptors.BCUT2D(mol)
        desc.update({
            'BCUT2D_LOGPLOW': bcut[5],
            'BCUT2D_LOGPHI': bcut[4],
            'BCUT2D_CHGHI': bcut[2],
            'BCUT2D_MRLOW': bcut[7],
            'BCUT2D_MWHI': bcut[0]
        })
    except:
        desc.update({k: None for k in ['BCUT2D_LOGPLOW', 'BCUT2D_LOGPHI', 
                                      'BCUT2D_CHGHI', 'BCUT2D_MRLOW', 'BCUT2D_MWHI']})

    # SlogP_VSA descriptors
    try:
        slogp_vsa = rdMolDescriptors.CalcSlogPVSA(mol)
        desc.update({
            'SlogP_VSA2': slogp_vsa[1],
            'SlogP_VSA4': slogp_vsa[3],
            'SlogP_VSA5': slogp_vsa[4]
        })
    except:
        desc.update({k: None for k in ['SlogP_VSA2', 'SlogP_VSA4', 'SlogP_VSA5']})

    # EState_VSA descriptors
    try:
        estate_vsa = rdMolDescriptors.CalcEStateVSA(mol)
        desc.update({
            'VSA_EState8': estate_vsa[7],
            'EState_VSA1': estate_vsa[0],
            'VSA_EState5': estate_vsa[4],
            'VSA_EState2': estate_vsa[1],
            'VSA_EState4': estate_vsa[3],
            'VSA_EState3': estate_vsa[2],
            'EState_VSA8': estate_vsa[7],
            'EState_VSA2': estate_vsa[1]
        })
    except:
        desc.update({k: None for k in ['VSA_EState8', 'EState_VSA1', 'VSA_EState5',
                                      'VSA_EState2', 'VSA_EState4', 'VSA_EState3',
                                      'EState_VSA8', 'EState_VSA2']})

    # SMR_VSA descriptors
    try:
        smr_vsa = rdMolDescriptors.CalcSMRVSA(mol)
        desc.update({
            'SMR_VSA1': smr_vsa[0],
            'SMR_VSA7': smr_vsa[6],
            'SMR_VSA5': smr_vsa[4]
        })
    except:
        desc.update({k: None for k in ['SMR_VSA1', 'SMR_VSA7', 'SMR_VSA5']})

    # PEOE_VSA descriptors
    try:
        peoe_vsa = rdMolDescriptors.CalcPEOEVSA(mol)
        desc.update({
            'PEOE_VSA14': peoe_vsa[13],
            'PEOE_VSA7': peoe_vsa[6],
            'PEOE_VSA10': peoe_vsa[9]
        })
    except:
        desc.update({k: None for k in ['PEOE_VSA14', 'PEOE_VSA7', 'PEOE_VSA10']})

    # Morgan fingerprints
    try:
        fp1 = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 1, 2048)
        fp3 = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 3, 2048)
        desc.update({
            'FpDensityMorgan1': fp1.GetNumOnBits()/2048,
            'FpDensityMorgan3': fp3.GetNumOnBits()/2048
        })
    except:
        desc.update({k: None for k in ['FpDensityMorgan1', 'FpDensityMorgan3']})

    # Basic descriptors
    basic_desc = {
        'MolLogP': Descriptors.MolLogP,
        'qed': QED.qed,
        'MinEStateIndex': Descriptors.MinEStateIndex,
        'TPSA': Descriptors.TPSA,
        'HallKierAlpha': Descriptors.HallKierAlpha,
        'NOCount': Descriptors.NOCount,
        'BertzCT': Descriptors.BertzCT,
        'MinAbsEStateIndex': Descriptors.MinAbsEStateIndex,
        'NumHeteroatoms': Descriptors.NumHeteroatoms,
        'Kappa3': GraphDescriptors.Kappa3,
        'NumRotatableBonds': Descriptors.NumRotatableBonds,
        'BalabanJ': GraphDescriptors.BalabanJ,
        'Kappa1': GraphDescriptors.Kappa1,
        'NumHAcceptors': Descriptors.NumHAcceptors
    }

    for name, func in basic_desc.items():
        try:
            desc[name] = func(mol)
        except:
            desc[name] = None

    return desc

# Read data and process
df = pd.read_excel('test.xlsx', sheet_name='Sheet1')

# Get descriptor keys from a dummy molecule
dummy_mol = Chem.MolFromSmiles('C')
descriptor_keys = list(compute_descriptors(dummy_mol).keys())

results = []
for smi in df['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        results.append(compute_descriptors(mol))
    else:
        results.append({k: None for k in descriptor_keys})

df_result = pd.DataFrame(results)
df_final = pd.concat([df, df_result], axis=1)
df_final.to_excel('descriptors_output.xlsx', index=False)

In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# Function to convert SMILES to molecular formula
def smiles_to_formula(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return rdMolDescriptors.CalcMolFormula(mol)
        else:
            return "Invalid SMILES"
    except Exception as e:
        return f"Error: {str(e)}"

# Load SMILES from Excel
df = pd.read_excel("test.xlsx")

if 'smiles' not in df.columns:
    raise ValueError("Excel file must contain a column named 'smiles'")

# Compute molecular formula for each SMILES
df['MolecularFormula'] = df['smiles'].apply(smiles_to_formula)

# Save to a new Excel file
df.to_excel("molecular_formulas.xlsx", index=False)

print("✅ Molecular formulas saved to 'molecular_formulas.xlsx'")


✅ Molecular formulas saved to 'molecular_formulas.xlsx'


In [4]:
import pandas as pd
import re

# Load the Excel file
df = pd.read_excel("molfor.xlsx")

# Detect the molecular formula column automatically
formula_col = None
for col in df.columns:
    if "formula" in col.lower():
        formula_col = col
        break

if not formula_col:
    raise ValueError("No molecular formula column found.")

# Function to extract element counts from formula
def extract_elements(formula):
    elements = {'C': 0, 'H': 0, 'N': 0, 'O': 0}
    # Regex to extract elements and their counts (e.g., C6, H12, etc.)
    matches = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    for element, count in matches:
        if element in elements:
            elements[element] = int(count) if count else 1
    return elements

# Apply the extraction function to each formula
counts = df[formula_col].apply(extract_elements)

# Convert to DataFrame
counts_df = pd.DataFrame(list(counts))

# Merge with original data
result_df = pd.concat([df, counts_df], axis=1)

# Save to Excel
result_df.to_excel("molfor_with_element_counts.xlsx", index=False)

print("✅ Element counts (C, H, N, O) extracted and saved to 'molfor_with_element_counts.xlsx'")


✅ Element counts (C, H, N, O) extracted and saved to 'molfor_with_element_counts.xlsx'


In [1]:
from rdkit import Chem
from rdkit.Chem import Draw
import pandas as pd
from IPython.display import display
import os

def generate_structure(smiles, output_folder, index):
    mol = Chem.MolFromSmiles(smiles.strip())
    if not mol:
        print(f"Invalid SMILES string at index {index}. Please check the format.")
        return None
    img = Draw.MolToImage(mol, size=(300, 300))
    img.save(os.path.join(output_folder, f"structure_{index}.png"))

def main():
    input_file = input("Enter the path to the Excel file containing SMILES strings: ").strip()
    output_folder = input("Enter the path to the output folder for images: ").strip()

    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    try:
        df = pd.read_excel(input_file)
        if 'SMILES' not in df.columns:
            print("Error: The Excel file must contain a 'SMILES' column.")
            return

        for idx, smiles in enumerate(df['SMILES'], start=1):
            generate_structure(smiles, output_folder, idx)

        print(f"Structures generated successfully in folder: {output_folder}")

    except FileNotFoundError:
        print("Error: Excel file not found. Please check the file path.")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()


Enter the path to the Excel file containing SMILES strings:  D:\finalhere\gaussian_data_for_installation\pythoncodes\sheetBDE.xlsx
Enter the path to the output folder for images:  D:\finalhere\gaussian_data_for_installation\pythoncodes\imagesforBDE


Structures generated successfully in folder: D:\finalhere\gaussian_data_for_installation\pythoncodes\imagesforBDE
